# Excess mortality in US vs Europe: assemble the dataset

In [1]:
import numpy as np
import pylab as plt
import pandas as pd

## US data

In [2]:
# Ariel's analysis for this project

us = pd.read_csv('data/us_states_merged_20240217.csv')

# 2020-02-03 to 2021-07-03
# 2021-07-04 to 2022-04-02
# 2022-04-03 to 2023-04-30

window1 = ((us.year==2020) & (us.time>=6))  | ((us.year==2021) & (us.time<=26))
window2 = ((us.year==2021) & (us.time>=27)) | ((us.year==2022) & (us.time<=13))
window3 = ((us.year==2022) & (us.time>=14)) | ((us.year==2023) & (us.time<=17))

us1 = us[window1][['state', 'excess_deaths']].groupby('state').sum().reset_index()
us2 = us[window2][['state', 'excess_deaths']].groupby('state').sum().reset_index()
us3 = us[window3][['state', 'excess_deaths']].groupby('state').sum().reset_index()

us = us1

us = us.rename(columns={'state': 'region', 'excess_deaths': 'excess20_21'})
us['excess21_22'] = us2['excess_deaths']
us['excess22_23'] = us3['excess_deaths']

us.loc[us['region'] == 'DC', 'region'] = 'District of Columbia'
us = us.drop(us[us.region=='United States'].index)
us['US'] = 1

us

,region,excess20_21,excess21_22,excess22_23,US
0,Alabama,14594.646154,10166.475,2864.582692,1
1,Alaska,806.323077,1635.125,1088.678846,1
2,Arizona,21072.415385,14881.300,5568.069231,1
3,Arkansas,7722.338462,6601.725,2890.048077,1
4,California,77367.400000,37815.025,25478.025000,1
5,Colorado,8352.276923,7919.175,3943.421154,1
6,Connecticut,6523.200000,2475.850,474.850000,1
7,District of Columbia,1764.615385,714.800,293.969231,1
8,Delaware,2050.384615,1408.125,976.755769,1
9,Florida,40727.307692,40712.300,11304.884615,1


In [3]:
# https://www.census.gov/data/tables/time-series/demo/popest/2020s-state-total.html

# https://www2.census.gov/programs-surveys/popest/tables/2020-2023/state/totals/NST-EST2023-POP.xlsx

pop = pd.read_excel('data/NST-EST2023-POP.xlsx', skiprows=2)

pop = pop[6:-8]

pop['Geographic Area'] = [s[1:] for s in pop['Geographic Area'].values]

pop = pop[['Geographic Area', 'Unnamed: 3']] # 2021 column
pop = pop.rename(columns={'Geographic Area': 'region', 'Unnamed: 3': 'population'})

# Merge
us = pd.merge(us, pop, on='region')

In [4]:
us['rel_excess20_21'] = us.excess20_21 / us.population * 100
us['rel_excess21_22'] = us.excess21_22 / us.population * 100
us['rel_excess22_23'] = us.excess22_23 / us.population * 100

In [5]:
us_state_to_abbrev = {
    "Alabama": "AL",
    "Alaska": "AK",
    "Arizona": "AZ",
    "Arkansas": "AR",
    "California": "CA",
    "Colorado": "CO",
    "Connecticut": "CT",
    "Delaware": "DE",
    "Florida": "FL",
    "Georgia": "GA",
    "Hawaii": "HI",
    "Idaho": "ID",
    "Illinois": "IL",
    "Indiana": "IN",
    "Iowa": "IA",
    "Kansas": "KS",
    "Kentucky": "KY",
    "Louisiana": "LA",
    "Maine": "ME",
    "Maryland": "MD",
    "Massachusetts": "MA",
    "Michigan": "MI",
    "Minnesota": "MN",
    "Mississippi": "MS",
    "Missouri": "MO",
    "Montana": "MT",
    "Nebraska": "NE",
    "Nevada": "NV",
    "New Hampshire": "NH",
    "New Jersey": "NJ",
    "New Mexico": "NM",
    "New York": "NY",
    "North Carolina": "NC",
    "North Dakota": "ND",
    "Ohio": "OH",
    "Oklahoma": "OK",
    "Oregon": "OR",
    "Pennsylvania": "PA",
    "Rhode Island": "RI",
    "South Carolina": "SC",
    "South Dakota": "SD",
    "Tennessee": "TN",
    "Texas": "TX",
    "Utah": "UT",
    "Vermont": "VT",
    "Virginia": "VA",
    "Washington": "WA",
    "West Virginia": "WV",
    "Wisconsin": "WI",
    "Wyoming": "WY",
    "District of Columbia": "DC",
}
    
# invert the dictionary
abbrev_to_us_state = dict(map(reversed, us_state_to_abbrev.items()))

In [6]:
# https://data.cdc.gov/Vaccinations/COVID-19-Vaccinations-in-the-United-States-Jurisdi/unsk-b7fc/data_preview

vac = pd.read_csv('data/COVID-19_Vaccinations_in_the_United_States_Jurisdiction_20240219.csv')

vac = vac[vac.Date == '10/02/2021'][['Location', 'Series_Complete_Pop_Pct']]

vac.Location = [abbrev_to_us_state[s] if s in abbrev_to_us_state else s for s in vac.Location]

vac = vac.rename(columns={'Location': 'region', 'Series_Complete_Pop_Pct': 'vacc'})

us = pd.merge(us, vac, on='region')

In [7]:
# https://www.bea.gov/data/gdp/gdp-state
# https://apps.bea.gov/regional/histdata/releases/0322gdpstate/index.cfm

gdp = pd.read_csv('data/SAGDP1__ALL_AREAS_1997_2021.csv')

gdp = gdp[gdp.Description == 'Current-dollar GDP (millions of current dollars)']

gdp = gdp[gdp.LineCode == 3]

gdp = gdp[['GeoName', '2021']]
          
gdp = gdp.rename(columns={'GeoName': 'region', '2021': 'gdp'})

us = pd.merge(us, gdp, on='region')

us.gdp = us.gdp.astype('float') / us.population * 1e6

us['gdp_ppp'] = us.gdp

In [8]:
# # PPP adjustement

# ppp = pd.read_csv('BEA rppp.csv', skiprows=3)[:-2]

# ppp = [ppp[ppp.GeoName == state]['2019'].values[0] for state in us.region]

# us.gdp /= (np.array(ppp) / 100)

In [9]:
# https://en.wikipedia.org/wiki/List_of_U.S._states_and_territories_by_income_inequality
# https://data.census.gov/table/ACSDT1Y2019.B19083?q=Gini&g=040XX00US51,50,11,12,53,54,10,15,16,13,19,17,18,40,41,44,45,01,42,48,04,49,05,02,46,47,08,09,06,30,33,34,31,32,37,38,35,36,39,22,23,20,21,26,27,24,25,28,29,72,55,56_010XX00US&moe=false&tp=true&tid=ACSDT1Y2019.B19083

gini = pd.read_csv("data/ACSDT1Y2019.B19083-2026-03-03T215237.csv")
gini = pd.DataFrame({"region": gini.values[2::2,0], "gini": gini.values[3::2,1] * 100})

us = pd.merge(us, gini, on="region")

In [10]:
us

,region,excess20_21,excess21_22,excess22_23,US,population,rel_excess20_21,rel_excess21_22,rel_excess22_23,vacc,gdp,gdp_ppp,gini
0,Alabama,14594.646154,10166.475,2864.582692,1,5050380.0,0.288981,0.201301,0.056720,42.7,48925.526396,48925.526396,47.41
1,Alaska,806.323077,1635.125,1088.678846,1,734923.0,0.109715,0.222489,0.148135,50.8,74797.087586,74797.087586,43.76
2,Arizona,21072.415385,14881.300,5568.069231,1,7272487.0,0.289755,0.204625,0.076563,51.2,56540.740465,56540.740465,45.91
3,Arkansas,7722.338462,6601.725,2890.048077,1,3028443.0,0.254994,0.217991,0.095430,45.8,47729.278709,47729.278709,47.5
4,California,77367.400000,37815.025,25478.025000,1,39145060.0,0.197643,0.096602,0.065086,59.1,85748.531232,85748.531232,48.66
5,Colorado,8352.276923,7919.175,3943.421154,1,5811596.0,0.143717,0.136265,0.067854,59.5,72603.360591,72603.360591,45.48
6,Connecticut,6523.200000,2475.850,474.850000,1,3603691.0,0.181014,0.068703,0.013177,68.9,82276.088599,82276.088599,50.24
7,District of Columbia,1764.615385,714.800,293.969231,1,669037.0,0.263755,0.106840,0.043939,60.3,227204.026085,227204.026085,51.15
8,Delaware,2050.384615,1408.125,976.755769,1,1004881.0,0.204043,0.140129,0.097201,57.9,80326.128168,80326.128168,45.09
9,Florida,40727.307692,40712.300,11304.884615,1,21830708.0,0.186560,0.186491,0.051784,57.4,56173.052198,56173.052198,48.08


## EU data

In [11]:
# https://raw.githubusercontent.com/dkobak/excess-mortality/main/excess-mortality-timeseries.csv
    
eu = pd.read_csv('data/excess-mortality-timeseries.csv')

eu_countries = [
    'Albania',
    'Austria',
    'Belgium',
    'Bosnia',
    'Bulgaria',
    'Croatia',
    'Cyprus',
    'Czechia',
    'Denmark',
    'Estonia',
    'Finland',
    'France',
    'Germany',
    'Greece',
    'Hungary',
    'Iceland',
    'Ireland',
    'Italy',
    'Kosovo',
    'Latvia',
    'Lithuania',
    'Luxembourg',
    'Malta',
    'Moldova',
    'Montenegro',
    'Netherlands',
    'North Macedonia',
    'Norway',
    'Poland',
    'Portugal',
    'Romania',
    'Russia',
    'Serbia',
    'Slovakia',
    'Slovenia',
    'Spain',
    'Sweden',
    'Switzerland',
    'Ukraine',
    'United Kingdom'
]

# # 2020-02-03 to 2021-07-03
# # 2021-07-04 to 2022-04-02
# # 2022-04-03 to 2023-04-30

window = ((eu.year==2020) & (eu.time>=6)) | ((eu.year==2021) & (eu.time<=26))
selection = eu.country_name.isin(eu_countries) & (eu.time_unit=='weekly') & window

window = ((eu.year==2020) & (eu.time>=2)) | ((eu.year==2021) & (eu.time<=6))
selection |= eu.country_name.isin(eu_countries) & (eu.time_unit=='monthly') & window

eu1 = eu[selection][['country_name', 'excess deaths']].groupby('country_name').sum().reset_index()

window = ((eu.year==2021) & (eu.time>=27)) | ((eu.year==2022) & (eu.time<=13))
selection = eu.country_name.isin(eu_countries) & (eu.time_unit=='weekly') & window

window = ((eu.year==2021) & (eu.time>=7)) | ((eu.year==2022) & (eu.time<=3))
selection |= eu.country_name.isin(eu_countries) & (eu.time_unit=='monthly') & window

eu2 = eu[selection][['country_name', 'excess deaths']].groupby('country_name').sum().reset_index()

window = ((eu.year==2022) & (eu.time>=14)) | ((eu.year==2023) & (eu.time<=17))
selection = eu.country_name.isin(eu_countries) & (eu.time_unit=='weekly') & window

window = ((eu.year==2022) & (eu.time>=4)) | ((eu.year==2023) & (eu.time<=4))
selection |= eu.country_name.isin(eu_countries) & (eu.time_unit=='monthly') & window

eu3 = eu[selection][['country_name', 'excess deaths']].groupby('country_name').sum().reset_index()

eu = eu1.rename(columns={'country_name': 'region', 'excess deaths': 'excess20_21'})

eu['excess21_22'] = eu2['excess deaths']
eu.loc[eu.region != 'Ukraine', 'excess22_23'] = eu3['excess deaths'].values

eu['US'] = 0

eu

,region,excess20_21,excess21_22,excess22_23,US
0,Albania,11028.2,5476.3,-1207.9,0
1,Austria,9233.6,6350.2,9345.4,0
2,Belgium,16729.4,4145.4,7192.5,0
3,Bosnia,13698.8,9034.5,134.1,0
4,Bulgaria,32740.0,35437.0,-1711.4,0
5,Croatia,9921.4,10419.4,2983.8,0
6,Cyprus,443.8,1208.9,322.8,0
7,Czechia,34085.4,8384.1,3308.2,0
8,Denmark,-1289.2,3107.9,2675.8,0
9,Estonia,1644.4,1954.8,922.2,0


In [12]:
# https://population.un.org/wpp/

# https://population.un.org/wpp/Download/Files/1_Indicators%20(Standard)/CSV_FILES/WPP2022_TotalPopulationBySex.zip
    
pop = pd.read_csv('data/WPP2022_TotalPopulationBySex.zip', low_memory=False)
pop = pop[(pop['Variant'] == 'Medium') & (pop['Time'] == 2021)]
pop = pop[['Location', 'PopTotal']]
pop = pop.rename(columns={'Location': 'region', 'PopTotal': 'population'})
pop['population'] *= 1000

# Rosstat, estimate for 1 Jan 2021    
pop = pd.concat(
    [pop, pd.DataFrame({'region': ['Russia'], 'population': [146_171_015]})],
    ignore_index=True
)
# pop = pop.append({'region': 'Russia', 'population': 146_171_015}, ignore_index=True)

# Ukrstat 2021, according to Wikipedia (this does not include Crimea) 
# Donetsk People's Republic population, according to Wikipedia
# Luhansk People's Republic population, according to Wikipedia
pop.loc[pop.region=='Ukraine', 'population'] = 41_167_336 - 2_300_000 - 1_500_000

# World Bank dataset
pop = pd.concat(
    [pop, pd.DataFrame({'region': ['Kosovo'], 'population': [1_700_000]})],
    ignore_index=True
)
# pop = pop.append({'region': 'Kosovo', 'population': 1_700_000}, ignore_index=True)
pop.loc[pop.region == 'Serbia', 'population'] -= 1_700_000

# Name shortening
pop.loc[pop.region=='Bosnia and Herzegovina', 'region'] = 'Bosnia'
pop.loc[pop.region=='Republic of Moldova', 'region'] = 'Moldova'

pop = pop[pop.region.isin(eu_countries)]

# Merge
eu = pd.merge(eu, pop, on='region')

In [13]:
eu['rel_excess20_21'] = eu.excess20_21 / eu.population * 100
eu['rel_excess21_22'] = eu.excess21_22 / eu.population * 100
eu['rel_excess22_23'] = eu.excess22_23 / eu.population * 100

In [14]:
# https://data.worldbank.org/indicator/NY.GDP.PCAP.CD

gdp = pd.read_csv('data/API_NY.GDP.PCAP.CD_DS2_en_csv_v2_184.csv', skiprows=4)

gdp.loc[gdp['Country Name']=='Bosnia and Herzegovina', 'Country Name'] = 'Bosnia'
gdp.loc[gdp['Country Name']=='Russian Federation', 'Country Name'] = 'Russia'
gdp.loc[gdp['Country Name']=='Slovak Republic', 'Country Name'] = 'Slovakia'

gdp = gdp[gdp['Country Name'].isin(eu_countries)][['Country Name', '2021']]
gdp = gdp.rename(columns={'Country Name': 'region', '2021': 'gdp'})

eu = pd.merge(eu, gdp, on='region')

In [15]:
# https://data.worldbank.org/indicator/NY.GDP.PCAP.PP.CD

gdp = pd.read_csv('data/API_NY.GDP.PCAP.PP.CD_DS2_en_csv_v2_9.csv', skiprows=4)

gdp.loc[gdp['Country Name']=='Bosnia and Herzegovina', 'Country Name'] = 'Bosnia'
gdp.loc[gdp['Country Name']=='Russian Federation', 'Country Name'] = 'Russia'
gdp.loc[gdp['Country Name']=='Slovak Republic', 'Country Name'] = 'Slovakia'

gdp = gdp[gdp['Country Name'].isin(eu_countries)][['Country Name', '2021']]
gdp = gdp.rename(columns={'Country Name': 'region', '2021': 'gdp_ppp'})

eu = pd.merge(eu, gdp, on='region')

In [16]:
# https://covid.ourworldindata.org/data/owid-covid-data.csv

vac = pd.read_csv('data/owid-covid-data.zip', low_memory=False)

vac.loc[vac.location=='Bosnia and Herzegovina', 'location'] = 'Bosnia'

vac = vac[(vac.date >= '2021-10-02') & (vac.location.isin(eu_countries))]

v1 = vac[['location', 'people_fully_vaccinated_per_hundred']].groupby('location').min().reset_index()
v2 = vac[['location', 'people_vaccinated_per_hundred']].groupby('location').min().reset_index()

v1.loc[v1.location=='Switzerland', 'people_fully_vaccinated_per_hundred'] = \
    v2[v2.location=='Switzerland']['people_vaccinated_per_hundred']

vac = v1.rename(columns={'location': 'region', 'people_fully_vaccinated_per_hundred': 'vacc'})

eu = pd.merge(eu, vac, on='region')

In [17]:
# https://data.worldbank.org/indicator/SI.POV.GINI

gini = pd.read_csv("data/API_SI.POV.GINI_DS2_en_csv_v2_47.csv", skiprows=4)
gini["gini"] = np.nanmax(gini.values[:,4:-1].astype(float), axis=1)
gini = gini.rename(columns={"Country Name": "region"})
gini = gini[["region", "gini"]]

eu = pd.merge(eu, gini, on="region")

/tmp/ipykernel_1368836/3279447760.py:4: RuntimeWarning: All-NaN slice encountered
  gini["gini"] = np.nanmax(gini.values[:,4:-1].astype(float), axis=1)


In [18]:
eu

,region,excess20_21,excess21_22,excess22_23,US,population,rel_excess20_21,rel_excess21_22,rel_excess22_23,gdp,gdp_ppp,vacc,gini
0,Albania,11028.2,5476.3,-1207.9,0,2854710.0,0.386316,0.191834,-0.042313,6377.203096,15471.056173,28.34,34.6
1,Austria,9233.6,6350.2,9345.4,0,8922082.0,0.103492,0.071174,0.104745,53517.890451,59828.882812,62.34,31.5
2,Belgium,16729.4,4145.4,7192.5,0,11611419.0,0.144077,0.035701,0.061943,51850.397184,59508.673870,72.87,33.1
3,Bulgaria,32740.0,35437.0,-1711.4,0,6885868.0,0.475467,0.514634,-0.024854,12219.341871,27975.174303,19.71,41.3
4,Croatia,9921.4,10419.4,2983.8,0,4060135.0,0.244361,0.256627,0.073490,17809.032390,35155.627182,40.54,33.7
5,Cyprus,443.8,1208.9,322.8,0,1244188.0,0.035670,0.097164,0.025945,32745.843750,46110.097656,62.15,37.0
6,Czechia,34085.4,8384.1,3308.2,0,10510751.0,0.324291,0.079767,0.031474,26822.514186,45630.037807,57.11,27.5
7,Denmark,-1289.2,3107.9,2675.8,0,5854240.0,-0.022022,0.053088,0.045707,69268.651798,66086.879455,72.71,29.9
8,Estonia,1644.4,1954.8,922.2,0,1328701.0,0.123760,0.147121,0.069406,27943.701220,43476.854226,54.95,39.4
9,Finland,549.0,4569.8,6662.4,0,5535992.0,0.009917,0.082547,0.120347,53504.693648,54778.269006,63.64,28.3


## Concatenate

In [19]:
df = pd.concat((us, eu))

In [20]:
df

,region,excess20_21,excess21_22,excess22_23,US,population,rel_excess20_21,rel_excess21_22,rel_excess22_23,vacc,gdp,gdp_ppp,gini
0,Alabama,14594.646154,10166.475,2864.582692,1,5050380.0,0.288981,0.201301,0.056720,42.70,48925.526396,48925.526396,47.41
1,Alaska,806.323077,1635.125,1088.678846,1,734923.0,0.109715,0.222489,0.148135,50.80,74797.087586,74797.087586,43.76
2,Arizona,21072.415385,14881.300,5568.069231,1,7272487.0,0.289755,0.204625,0.076563,51.20,56540.740465,56540.740465,45.91
3,Arkansas,7722.338462,6601.725,2890.048077,1,3028443.0,0.254994,0.217991,0.095430,45.80,47729.278709,47729.278709,47.5
4,California,77367.400000,37815.025,25478.025000,1,39145060.0,0.197643,0.096602,0.065086,59.10,85748.531232,85748.531232,48.66
...,...,...,...,...,...,...,...,...,...,...,...,...,...
32,Spain,81017.800000,20516.300,41285.000000,0,47486935.0,0.170611,0.043204,0.086940,77.26,30488.820953,41182.459621,36.5
33,Sweden,8592.400000,2393.400,4918.700000,0,10467097.0,0.082090,0.022866,0.046992,62.24,61417.680877,60396.713701,31.6
34,Switzerland,8969.200000,3997.000,5652.700000,0,8691406.0,0.103196,0.045988,0.065038,63.77,93446.434452,77181.375348,36.0
35,Ukraine,93526.200000,89397.000,NaN,0,37367336.0,0.250289,0.239238,NaN,14.55,4827.845703,14289.040039,39.2


In [21]:
df.to_csv('us-vs-eu-data.csv', index=False, float_format='%.3f')